# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [ ]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [15]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/kaylahiltermann/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/kaylahiltermann/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [16]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [17]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [18]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [20]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [21]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [22]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [23]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [24]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node 'c97564'. Skipping!
Property 'summary' already exists in node 'ae2cdb'. Skipping!
Property 'summary' already exists in node 'af28ad'. Skipping!
Property 'summary' already exists in node '202123'. Skipping!
Property 'summary' already exists in node '6d577d'. Skipping!
Property 'summary' already exists in node '7127a9'. Skipping!
Property 'summary' already exists in node '9ce613'. Skipping!
Property 'summary' already exists in node '5ae1c2'. Skipping!
Property 'summary' already exists in node 'b4b954'. Skipping!
Property 'summary' already exists in node '9e20a5'. Skipping!
Property 'summary' already exists in node 'e4a269'. Skipping!
Property 'summary' already exists in node '29a4f6'. Skipping!
Property 'summary' already exists in node 'a1a7ed'. Skipping!
Property 'summary' already exists in node 'a6c546'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/43 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'c97564'. Skipping!
Property 'summary_embedding' already exists in node 'ae2cdb'. Skipping!
Property 'summary_embedding' already exists in node '7127a9'. Skipping!
Property 'summary_embedding' already exists in node '9e20a5'. Skipping!
Property 'summary_embedding' already exists in node '6d577d'. Skipping!
Property 'summary_embedding' already exists in node '202123'. Skipping!
Property 'summary_embedding' already exists in node 'af28ad'. Skipping!
Property 'summary_embedding' already exists in node 'b4b954'. Skipping!
Property 'summary_embedding' already exists in node '5ae1c2'. Skipping!
Property 'summary_embedding' already exists in node 'e4a269'. Skipping!
Property 'summary_embedding' already exists in node 'a6c546'. Skipping!
Property 'summary_embedding' already exists in node '9ce613'. Skipping!
Property 'summary_embedding' already exists in node 'a1a7ed'. Skipping!
Property 'summary_embedding' already exists in node '29a4f6'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 40, relationships: 480)

We can save and load our knowledge graphs as follows.

In [25]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 40, relationships: 480)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [26]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [27]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.


Finally, we can use our `TestSetGenerator` to generate our testset!

### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [ ]:
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
testset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)
testset.to_pandas()

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node 'a679be'. Skipping!
Property 'summary' already exists in node '725c2e'. Skipping!
Property 'summary' already exists in node '38c3f2'. Skipping!
Property 'summary' already exists in node 'c5a1c2'. Skipping!
Property 'summary' already exists in node '0af6df'. Skipping!
Property 'summary' already exists in node '180d30'. Skipping!
Property 'summary' already exists in node '7346ae'. Skipping!
Property 'summary' already exists in node '2950b0'. Skipping!
Property 'summary' already exists in node 'e1589f'. Skipping!
Property 'summary' already exists in node '3dbdc3'. Skipping!
Property 'summary' already exists in node 'd77d06'. Skipping!
Property 'summary' already exists in node '4eea81'. Skipping!
Property 'summary' already exists in node 'e41741'. Skipping!
Property 'summary' already exists in node 'd2bc49'. Skipping!


...

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is the role of the School Participation D...,"[Chapter 1 Academic Years, Academic Calendars,...",The context does not specify the exact role of...,single_hop_specifc_query_synthesizer
1,What is 34 CFR 668.3(b) about in instructional...,[Regulatory Citations Academic year minimums: ...,34 CFR 668.3(b) relates to weeks of instructio...,single_hop_specifc_query_synthesizer
2,Volume 8 include clinical work?,[Inclusion of Clinical Work in a Standard Term...,Inclusion of Clinical Work in a Standard Term ...,single_hop_specifc_query_synthesizer
3,Is the Federal Work-Study program subject to p...,[Non-Term Characteristics A program that measu...,"No, the payment period is applicable to all Ti...",single_hop_specifc_query_synthesizer
4,Whts the instrctional time and weeks of instrc...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The academic year requirements specify that fo...,multi_hop_abstract_query_synthesizer
5,regulations approval process academic year 34 ...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The regulations specify that the academic year...,multi_hop_abstract_query_synthesizer
6,Academic years and regulatory citations like 3...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The context explains that academic years must ...,multi_hop_abstract_query_synthesizer
7,How do regs and approval process for academic ...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Regulations require that each eligible program...,multi_hop_abstract_query_synthesizer
8,Volume 8 and Volume 7 include clinical work in...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The context explains that clinical work includ...,multi_hop_specific_query_synthesizer
9,How do Appendix A and Appendix B collectively ...,[<1-hop>\n\nDisbursement Timing in Subscriptio...,Appendix B offers detailed guidance on the dis...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [31]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [33]:
for data_row in testset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [34]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [35]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [36]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [37]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [38]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [39]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [40]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [41]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [42]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'The kinds of loans available include:\n\n- Direct Subsidized Loans  \n- Direct Unsubsidized Loans  \n- Direct PLUS Loans (student Federal PLUS Loans and parent PLUS Loans on behalf of dependent students)  \n- Subsidized and Unsubsidized Federal Stafford Loans (made under the Federal Family Education Loan (FFEL) Program before July 1, 2010)  \n- Federal SLS Loans  \n- Federal PLUS Loans (made under the FFEL Program before July 1, 2010)\n\nNote that no new FFEL Program loans have been made since July 1, 2010, and graduate or professional students are only eligible for Direct Unsubsidized Loans (not Direct Subsidized Loans).'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [43]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [ ]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate, EvaluationResult

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

# Custom numerical empathy evaluator
def empathy_evaluator(run, example):
    """Custom evaluator that returns numerical empathy score from 1-5"""
    
    empathy_prompt = f"""
Please evaluate the empathy level of the following response on a scale of 1 to 5, where:
- 1 = Not empathetic at all, cold or dismissive
- 2 = Slightly empathetic, minimal acknowledgment of user's feelings
- 3 = Moderately empathetic, some understanding shown
- 4 = Very empathetic, clear acknowledgment and understanding
- 5 = Extremely empathetic, makes user feel truly heard and understood

Question: {example.inputs["question"]}
Response: {run.outputs["output"]}

Please respond with ONLY the numerical score (1-5) and nothing else.
"""
    
    # Get the LLM response
    llm_response = eval_llm.invoke(empathy_prompt)
    
    # Parse the numerical score
    try:
        score = int(llm_response.content.strip())
        # Ensure score is within valid range
        if 1 <= score <= 5:
            return EvaluationResult(
                key="empathy",
                score=score,
                value=score,
                comment=f"Empathy score: {score}/5"
            )
        else:
            return EvaluationResult(
                key="empathy",
                score=3,  # Default to middle score if invalid
                value=3,
                comment=f"Invalid score returned: {score}, defaulting to 3"
            )
    except (ValueError, AttributeError):
        return EvaluationResult(
            key="empathy",
            score=3,  # Default to middle score if parsing fails
            value=3,
            comment=f"Failed to parse score from: {llm_response.content}, defaulting to 3"
        )

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `empathy_evaluator`:

## LangSmith Evaluation

In [56]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'damp-land-29' at:
https://smith.langchain.com/o/4e8d6936-58a0-4ff9-bfae-6484988e5784/datasets/3a043ef0-bfb0-425e-96f4-c505a508f131/compare?selectedSessions=87241ad7-4275-4003-a5e9-ed60adca8071




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Volume 2 and Volume 7 relate to the aca...,I don't know.,None,Volume 2 discusses the requirements for defini...,0,0,0,1.356970,ceef55a8-3bf5-4be9-bcdf-b382ce358e4f,ac3cf6cc-63a4-4f02-9116-bd1ea591fd11
1,Chapter 2 include clinical work in standard te...,Based on the provided context:\n\n- Chapter 2 ...,None,Chapter 2 explains that clinical work outside ...,1,1,0,13.148940,ddb4e51d-2ebd-4b5d-b31d-dd18a501bee3,fcf5eacb-83ba-45e7-9fbd-2e856ff80d26
2,How do Appendix A and Appendix B collectively ...,I don't know.,None,Appendix B offers detailed guidance on the dis...,0,0,0,0.986089,5970eae8-99f3-4e4d-8379-331a5482c797,fee097bb-5a0f-4712-892e-a5024728eb05
3,Volume 8 and Volume 7 include clinical work in...,"Based on the provided context, clinical work t...",None,The context explains that clinical work includ...,1,1,0,5.897675,37156914-1a28-4cf6-90d3-85e362e202a7,e80b02e0-3674-46ab-b82a-6799fb85e1e9
4,How do regs and approval process for academic ...,"Based on the provided context, the regulations...",None,Regulations require that each eligible program...,1,1,0,7.419523,8d602c34-e337-4396-bc34-4933e4691f3d,5f4346d5-0b3a-4ac9-a1cc-e27ae62764fb
5,Academic years and regulatory citations like 3...,"Based on the provided context, academic years ...",None,The context explains that academic years must ...,1,0,0,4.090785,0e538cab-1e69-4cca-81f4-c777e9b989e3,f9ef2f7c-ad09-43ac-ac05-9ea4fb6fda4d
6,regulations approval process academic year 34 ...,I don't know,None,The regulations specify that the academic year...,0,0,0,1.018556,a5ff298e-7da6-474d-9799-84c038f37765,581ef6d5-5d36-4792-b9c7-88c613aa7f2e
7,Whts the instrctional time and weeks of instrc...,"Instructional time excludes scheduled breaks, ...",None,The academic year requirements specify that fo...,1,1,0,6.304342,b3f35830-72c8-489d-bb28-71276e88ac88,b4906479-cea6-481e-bf4b-e6972c7659b2
8,Is the Federal Work-Study program subject to p...,"No, the Federal Work-Study (FWS) Program is no...",None,"No, the payment period is applicable to all Ti...",1,1,0,1.727074,df9cc47f-dced-4b1e-a5de-d89180aeaca0,25604dd8-a3f2-4b01-a6bc-ef3774a90123
9,Volume 8 include clinical work?,"Yes, Volume 8 includes guidance on clinical wo...",None,Inclusion of Clinical Work in a Standard Term ...,1,0,0,2.082520,e7922069-c1fe-4792-baa9-4ac29ec498f8,c6055493-a5ca-4101-8ad1-56b900ffc614


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [46]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [47]:
rag_documents = docs

In [48]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

In [49]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

In [50]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [51]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [52]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [53]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

'Thank you for your question. Based on the information you’ve shared, there are several types of federal student loans available to help cover the cost of attendance:\n\n1. **Direct Subsidized Loans** – These are loans for students with demonstrated financial need. The government pays the interest while you’re in school at least half-time.\n\n2. **Direct Unsubsidized Loans** – These loans are available to both dependent and independent students regardless of financial need, but interest accrues while you’re in school.\n\n3. **Direct PLUS Loans** – These loans are available to parents of dependent students (Direct PLUS Loans for parents) or to graduate/professional students (Direct PLUS Loans for students). They can cover the student’s cost of attendance minus other financial aid, but eligibility requirements apply.\n\nIt’s also mentioned that if a parent is unable to obtain a Direct PLUS Loan, dependent students can receive additional Direct Unsubsidized Loan funds under independent st

Finally, we can evaluate the new chain on the same test set!

In [54]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'fresh-copy-61' at:
https://smith.langchain.com/o/4e8d6936-58a0-4ff9-bfae-6484988e5784/datasets/3a043ef0-bfb0-425e-96f4-c505a508f131/compare?selectedSessions=115a535d-a79d-4aa6-9c0e-c6a8c8e91fd8




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Volume 2 and Volume 7 relate to the aca...,Thank you for your thoughtful question. Based ...,None,Volume 2 discusses the requirements for defini...,0,0,1,2.294316,ceef55a8-3bf5-4be9-bcdf-b382ce358e4f,fe410289-17c4-45db-8377-174705cbcf47
1,Chapter 2 include clinical work in standard te...,Thank you for your thoughtful question. It’s c...,None,Chapter 2 explains that clinical work outside ...,1,1,1,5.466743,ddb4e51d-2ebd-4b5d-b31d-dd18a501bee3,ca0e556b-96b6-4f24-a4ff-4d8907627d05
2,How do Appendix A and Appendix B collectively ...,Thank you for your thoughtful question. Based ...,None,Appendix B offers detailed guidance on the dis...,0,0,1,5.513448,5970eae8-99f3-4e4d-8379-331a5482c797,b9eedcf4-0ea7-4c57-918c-1a06b6da0a98
3,Volume 8 and Volume 7 include clinical work in...,Thank you for your thoughtful question. Based ...,None,The context explains that clinical work includ...,1,1,1,6.404129,37156914-1a28-4cf6-90d3-85e362e202a7,712701ed-3df0-47b5-8d42-b4b7cf6a6533
4,How do regs and approval process for academic ...,Thank you for your thoughtful question. It’s c...,None,Regulations require that each eligible program...,1,1,1,6.707642,8d602c34-e337-4396-bc34-4933e4691f3d,c7649870-9d93-4956-8519-319e1c5f9f7b
5,Academic years and regulatory citations like 3...,Thank you for your thoughtful question. Based ...,None,The context explains that academic years must ...,1,0,1,4.234408,0e538cab-1e69-4cca-81f4-c777e9b989e3,22599ae9-df62-44cd-9e50-5e3269074cfd
6,regulations approval process academic year 34 ...,Thank you for your question. From the context ...,None,The regulations specify that the academic year...,1,1,1,4.651403,a5ff298e-7da6-474d-9799-84c038f37765,48bd71c1-ca1b-4d30-932d-5a0d4ca49b3a
7,Whts the instrctional time and weeks of instrc...,Thank you for your thoughtful question. Based ...,None,The academic year requirements specify that fo...,1,0,1,8.114091,b3f35830-72c8-489d-bb28-71276e88ac88,304386d4-8d2f-4636-8278-d4301b2b4f6c
8,Is the Federal Work-Study program subject to p...,Thank you for your thoughtful question. Based ...,None,"No, the payment period is applicable to all Ti...",1,1,1,8.398972,df9cc47f-dced-4b1e-a5de-d89180aeaca0,081b313b-c6fb-4217-8012-ba1e0d14e2c1
9,Volume 8 include clinical work?,Thank you for your question. Based on the cont...,None,Inclusion of Clinical Work in a Standard Term ...,1,1,1,2.922058,e7922069-c1fe-4792-baa9-4ac29ec498f8,c226841b-6444-4a37-813e-5077e0124f67


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.